## はじめに

このレッスンでは以下を扱います：
- 関数呼び出しとは何かとそのユースケース
- Azure OpenAI を使った関数呼び出しの作成方法
- 関数呼び出しをアプリケーションに統合する方法

## 学習目標

このレッスンを修了すると、以下を理解し使いこなせるようになります：

- 関数呼び出しを使う目的
- Azure Open AI サービスで関数呼び出しをセットアップする方法
- アプリケーションのユースケースに適した効果的な関数呼び出しの設計


## 関数呼び出しの理解

このレッスンでは、教育系スタートアップのために、ユーザーがチャットボットを使って技術コースを探せる機能を構築したいと思います。ユーザーのスキルレベル、現在の役割、関心のある技術に合ったコースをおすすめします。

これを実現するために以下の組み合わせを使います：
 - ユーザーのためにチャット体験を作成する `Azure Open AI`
 - ユーザーのリクエストに基づいてコースを検索するのを助ける `Microsoft Learn Catalog API`
 - ユーザーのクエリを受け取り、APIリクエストを行う関数に送るための `Function Calling`

まずは、なぜそもそも関数呼び出しを使いたいのかを見てみましょう：


### なぜFunction Callingなのか 

このコースの他のレッスンを完了していれば、Large Language Models（LLM）を使うことの強力さはおそらく理解できているでしょう。また、いくつかの限界も見えてきているかもしれません。 

Function CallingはAzure Open AI Serviceの機能で、以下の制約を克服するためのものです： 
1) 一貫した応答フォーマット 
2) チャットの文脈でアプリケーションの他のソースのデータを利用する能力 

Function Callingがなかった時点では、LLMからの応答は構造化されておらず一貫性がありませんでした。開発者は応答のバリエーションごとに対応できるよう複雑な検証コードを書く必要がありました。 

「ストックホルムの現在の天気は？」といった質問に対してはユーザーは回答を得られませんでした。これはモデルがトレーニングされた時点のデータに制限されていたためです。 

以下の例でこの問題を見てみましょう： 

たとえば、学生データのデータベースを作成して適切なコースを提案したいとします。以下には、非常に似た情報を含む二人の学生の説明があります。 


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


私たちはこれをLLMに送ってデータを解析してもらいたいと思います。これにより、後でアプリケーションでAPIに送信したり、データベースに保存したりすることができます。 

興味のある情報についてLLMに指示する2つの同一のプロンプトを作成しましょう： 


これをLLMに送って、当社の製品にとって重要な部分を解析させたいと思います。そこで、LLMに指示するために2つの同じプロンプトを作成できます： 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


これら二つのプロンプトを作成した後、`client.responses.create`を使用してそれらをLLMに送信します。プロンプトは`input`変数に格納し、役割を`user`に割り当てます。これは、ユーザーからのメッセージがチャットボットに書き込まれていることを模倣するためです。 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

これで、両方のリクエストをLLMに送信し、受け取った応答を確認できます。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



プロンプトが同じで説明も似ているにもかかわらず、`Grades` プロパティのフォーマットが異なる場合があります。

上記のセルを複数回実行すると、フォーマットが `3.7` または `3.7 GPA` になることがあります。

これは、LLMが書かれたプロンプトという非構造化データを受け取り、同じく非構造化データを返すためです。データを保存または使用する際に何を期待すべきかを明確にするために、構造化フォーマットが必要です。

関数呼び出し（functional calling）を使用することで、構造化されたデータを確実に受け取ることができます。関数呼び出しを使う場合、LLMは実際に関数を呼び出したり実行したりしません。代わりに、LLMが応答のために従うべき構造を作成します。その後、その構造化された応答を使用してアプリケーション内でどの関数を実行するかを決定します。  
 


![関数呼び出しフローダイヤグラム](../../../../translated_images/ja/Function-Flow.083875364af4f4bb.webp)


関数から返されたものを受け取り、それをLLMに返送します。LLMはその後、自然言語を使ってユーザーの質問に答えます。


### 関数呼び出しの使用例 

<strong>外部ツールの呼び出し</strong> 
チャットボットはユーザーの質問に答えるのに優れています。関数呼び出しを使用することで、チャットボットはユーザーからのメッセージを使って特定のタスクを完了できます。例えば、学生が「この科目でさらに助けが必要だと指導教員にメールを送って」とチャットボットに依頼すると、 `send_email(to: string, body: string)` 関数呼び出しを行うことができます。


**APIまたはデータベースクエリの作成**
ユーザーは自然言語を使って情報を見つけ、それがフォーマットされたクエリやAPIリクエストに変換されます。例として、教師が「最後の課題を完了した学生は誰か」とリクエストした場合、 `get_completed(student_name: string, assignment: int, current_status: string)` という関数が呼ばれる可能性があります。


<strong>構造化データの作成</strong>
ユーザーはテキストやCSVのブロックからLLMを使って重要な情報を抽出することができます。例えば、学生が平和協定に関するWikipediaの記事をAIフラッシュカードに変換することができます。これは `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` という関数を使って行うことができます。


## 2. 最初の関数呼び出しの作成 

関数呼び出しを作成するプロセスは、主に3つのステップに分かれます： 
1. 関数のリストとユーザーメッセージを用いてChat Completions APIを呼び出す 
2. モデルの応答を読み取り、関数やAPI呼び出しを実行するなどのアクションを行う 
3. 関数からの応答を使ってユーザーへの応答を作成するために、Chat Completions APIを再度呼び出す 


![関数呼び出しの流れ](../../../../translated_images/ja/LLM-Flow.3285ed8caf4796d7.webp)


### 関数呼び出しの要素

#### ユーザー入力

最初のステップはユーザーメッセージを作成することです。これはテキスト入力の値を動的に割り当てることも、ここで値を割り当てることもできます。Chat Completions API を使うのが初めての場合は、メッセージの `role` と `content` を定義する必要があります。

`role` は `system`（ルール作成）、`assistant`（モデル）、または `user`（エンドユーザー）のいずれかになります。関数呼び出しの場合、これを `user` として割り当て、例の質問を設定します。


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 関数を作成する。 

次に、関数とそのパラメーターを定義します。ここでは `search_courses` という関数を1つだけ使いますが、複数の関数を作成することもできます。

<strong>重要</strong> : 関数はシステムメッセージに含まれ、利用可能なトークン数にカウントされます。 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

`name` - 呼び出したい関数の名前。 

`description` - 関数の動作についての説明。ここでは具体的かつ明確であることが重要。 

`parameters` - モデルに応答で生成してほしい値と形式のリスト。 


`type` - プロパティが格納されるデータ型。 

`properties` - モデルが応答で使用する特定の値のリスト。 


`name` - モデルがフォーマットされた応答で使用するプロパティの名前。 

`type` - このプロパティのデータ型。 

`description` - 特定のプロパティの説明。 


<strong>オプション</strong>

`required` - 関数呼び出しを完了するために必要なプロパティ。 


### 関数呼び出しの作成 
関数を定義した後は、Chat Completion APIへの呼び出しにその関数を含める必要があります。これを行うには、リクエストに `functions` を追加します。この場合は `functions=functions` です。 

また、`function_call` を `auto` に設定するオプションもあります。これは、関数の割り当てを自分で行うのではなく、ユーザーメッセージに基づいてどの関数を呼び出すべきかをLLMに判断させることを意味します。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


では、レスポンスを見てどのようにフォーマットされているか確認しましょう： 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

関数の名前が呼び出されているのがわかり、ユーザーのメッセージから、LLMが関数の引数に合うデータを見つけることができました。 


## 3.アプリケーションへの関数呼び出しの統合 


LLMからの整形された応答をテストした後、これをアプリケーションに統合できます。 

### フローの管理 

これをアプリケーションに統合するために、次の手順を踏みましょう。 

まず、Open AIサービスへの呼び出しを行い、応答を`response_message`という変数に格納します。 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


次に、Microsoft Learn API を呼び出してコースの一覧を取得する関数を定義します： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



ベストプラクティスとして、まずモデルが関数を呼び出そうとしているかどうかを確認します。その後、利用可能な関数の中から呼び出されている関数に一致するものを作成します。 
次に、関数の引数を取り、その引数をLLMの引数にマッピングします。

最後に、関数呼び出しメッセージと `search_courses` メッセージによって返された値を追加します。これにより、LLMはユーザーに自然言語で応答するために必要なすべての情報を得ることができます。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


これで、更新されたメッセージをLLMに送信し、APIのJSON形式の応答ではなく自然言語での応答を受け取ることができます。 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## コードチャレンジ 

素晴らしい！Azure Open AI Function Callingの学習を続けるために、以下を構築できます：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 学習者がより多くのコースを見つけるのに役立つ可能性のある関数の追加パラメーター。利用可能なAPIパラメーターはここで確認できます： 
 - 学習者の母国語など、より多くの情報を取得する別の関数呼び出しを作成する 
 - 関数呼び出しやAPI呼び出しで適切なコースが返されなかった場合のエラーハンドリングを作成する 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
